# Fine-tuning Text-to-SQL: Qwen3.5-4B với Unsloth trên Google Colab

Notebook 1-Click: Huấn luyện QLoRA 4-bit mô hình **Qwen3.5-4B**, áp dụng **Loss Masking** (`train_on_responses_only`) và xuất file **GGUF (Q4_K_M)** lưu thẳng về Google Drive.

In [ ]:
# 1. Kiểm tra GPU và cài đặt Unsloth
!nvidia-smi
!pip install unsloth unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes datasets


In [ ]:
# 2. Mount Google Drive để lưu checkpoint và file GGUF an toàn
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/Finetune_Text2SQL'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Thư mục lưu trữ: {SAVE_DIR}')

In [ ]:
# 3. Nạp mô hình Qwen3.5-4B ở dạng 4-bit (NF4)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'Qwen/Qwen3.5-4B',
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print('Nạp Base Model 4-bit thành công!')

In [ ]:
# 4. Gắn LoRA Adapter (Target toàn bộ 7 linear modules)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)
model.print_trainable_parameters()

In [ ]:
# 5. Nạp tập dữ liệu train.jsonl
from datasets import load_dataset

DATA_PATH = 'train.jsonl'
if not os.path.exists(DATA_PATH):
    drive_data = os.path.join(SAVE_DIR, 'train.jsonl')
    if os.path.exists(drive_data):
        DATA_PATH = drive_data
    else:
        print(f'Vui lòng upload train.jsonl vào session Colab hoặc đặt tại: {drive_data}')

dataset = load_dataset('json', data_files=DATA_PATH, split='train')
print('Số lượng mẫu:', len(dataset))

In [ ]:
# 6. Định dạng Chat Template và cấu hình Loss Masking
def formatting_prompts_func(examples):
    convos = examples['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { 'text' : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.05,
        num_train_epochs = 1,
        learning_rate = 1.5e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'cosine',
        seed = 3407,
        output_dir = os.path.join(SAVE_DIR, 'checkpoints'),
        report_to = 'none',
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part = '<|im_start|>assistant\n',
)

In [ ]:
# 7. Huấn luyện!
trainer_stats = trainer.train()
print('Huấn luyện hoàn tất!')

In [ ]:
# 8. Test suy luận thử nghiệm
FastLanguageModel.for_inference(model)

test_prompt = '''<|im_start|>system
You are a SQLite expert. Given the database schema, write the correct SQL query.
### DATABASE SCHEMA:
CREATE TABLE customers (id INT PRIMARY KEY, name TEXT, tier TEXT);
CREATE TABLE orders (order_id INT, customer_id INT, amount REAL, FOREIGN KEY(customer_id) REFERENCES customers(id));
<|im_end|>
<|im_start|>user
Find total order amount for VIP customers having total spent over 500?
<|im_end|>
<|im_start|>assistant
'''

inputs = tokenizer([test_prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 9. Xuất GGUF Q4_K_M lưu về Google Drive hoặc Push lên Hugging Face
# Cách A: Lưu trực tiếp vào Google Drive
gguf_output = os.path.join(SAVE_DIR, 'qwen3_5_4b_text2sql_gguf')
model.save_pretrained_gguf(gguf_output, tokenizer, quantization_method='q4_k_m')
print(f'File GGUF đã lưu tại Drive: {gguf_output}')

# Cách B (Tùy chọn): Push thẳng lên Hugging Face Hub (GGUF)
# from huggingface_hub import login
# login('hf_your_write_token_here')
# model.push_to_hub_gguf('your_username/Qwen3.5-4B-Text2SQL-GGUF', tokenizer, quantization_method='q4_k_m')
